# EBHI-SEG external pilot evaluation

Evaluate the frozen final ResNet50 on a deterministic balanced subset of 50 Normal and 50 Adenocarcinoma EBHI-SEG source images. No training, tuning, mask use, or preprocessing changes are performed. This is preliminary external evidence, not clinical validation.

In [8]:
from pathlib import Path
import shutil
import zipfile

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_auc_score,
)

CONTENT = Path("/content")
MODEL_PATH = CONTENT / "resnet50_binary_best.keras"
ZIP_PATH = CONTENT / "EBHI-SEG-pilot-colab.zip"
EXTRACT_ROOT = CONTENT / "ebhi_external_test"
assert MODEL_PATH.is_file(), f"Model not found: {MODEL_PATH}"
assert ZIP_PATH.is_file(), f"Pilot ZIP not found: {ZIP_PATH}"
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(EXTRACT_ROOT)

normal_files = sorted((EXTRACT_ROOT / "Normal" / "image").glob("*.png"))
cancer_files = sorted((EXTRACT_ROOT / "Adenocarcinoma" / "image").glob("*.png"))
assert len(normal_files) == len(cancer_files) == 50
paths = normal_files + cancer_files
y_true = np.array([0] * 50 + [1] * 50, dtype=np.int32)
print("Normal images:", len(normal_files))
print("Adenocarcinoma images:", len(cancer_files))
print("Total:", len(paths))

bgr_means = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
def load_and_preprocess(path):
    image = tf.io.decode_png(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, (224, 224), method="nearest")
    return tf.reverse(tf.cast(image, tf.float32), axis=[-1]) - bgr_means

dataset = (
    tf.data.Dataset.from_tensor_slices([str(path) for path in paths])
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
y_probability = model.predict(dataset, verbose=1).ravel()
y_pred = (y_probability >= 0.5).astype(np.int32)
print("\nEBHI-SEG EXTERNAL TEST")
print("======================")
print(f"Accuracy:          {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"ROC-AUC:           {roc_auc_score(y_true, y_probability):.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Adenocarcinoma"], digits=4))

Normal images: 50
Adenocarcinoma images: 50
Total: 100
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 4s/step

EBHI-SEG EXTERNAL TEST
Accuracy:          0.8900
Balanced accuracy: 0.8900
ROC-AUC:           1.0000

Confusion matrix:
[[39 11]
 [ 0 50]]

Classification report:
                precision    recall  f1-score   support

        Normal     1.0000    0.7800    0.8764        50
Adenocarcinoma     0.8197    1.0000    0.9009        50

      accuracy                         0.8900       100
     macro avg     0.9098    0.8900    0.8887       100
  weighted avg     0.9098    0.8900    0.8887       100


## Interpretation and limitations

Accuracy fell to 89% under the unchanged 0.5 threshold: all 50 adenocarcinoma images were detected, while 11 of 50 normal images were false positives. ROC-AUC of 1.0 shows complete ranking separation on this pilot, consistent with cross-dataset probability shift rather than loss of discrimination. The threshold was not adjusted because tuning and evaluating it on these same images would leak test information. The 100-image pilot is research evidence only and is not clinical validation.